# TALSIM KOSTRA Results Analyzer

**Purpose:** Reads and evaluates the `.WEL` output files produced by TALSIM-NG
for all KOSTRA design-storm scenarios and extracts peak flows, flood volumes,
and water levels for each return period and storm duration.

**What it does:**
- Parses TALSIM `.WEL` result files (fixed-width, Latin-1 encoded)
- Extracts time series for inflow, outflow, and reservoir storage
- Builds summary tables of peak values per scenario
- Produces multi-panel publication-ready plots

**User settings:** Change only the two clearly marked lines at the top  
**Input:** TALSIM `.WEL` output files  
**Output:** Summary tables, peak-flow plots

---

In [ ]:
# =============================================================================
# TALSIM WEL Results Analyzer
# =============================================================================
# HOW TO USE:
#   Only change the 2 lines marked below, then run the script.
#   Everything else is automatic.
# =============================================================================

# %% Imports & Setup
from pathlib import Path
import re
import unicodedata
import datetime as dt
from functools import reduce

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

plt.rcParams["figure.dpi"] = 200
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.axisbelow"] = True
plt.rcParams["font.size"] = 10
plt.rcParams["legend.frameon"] = False

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

# =============================================================================
# *** CHANGE ONLY THESE 2 LINES ***
# =============================================================================

MAIN_FOLDER = Path(r"C:\Users\raah\Desktop\Project_ZR\Ziegenrück_nr\Ziegenrück_41.78_15min")
WEL_NAME    = "Ziegenrück_nr.WEL"

# =============================================================================
# Everything below is fixed - no need to change
# =============================================================================

# Column to extract from the WEL file (inflow to retention basin)
VALUE_COL = "S020_1ZU"

# All return periods [years] that can appear in subfolder names
JAEHRLICHKEITEN_YR = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000, 10000]

# Optional: Excel with catchment area [km2] for specific runoff (set to None to skip)
EXCEL_PARAM_PATH = None  # e.g. Path("KOSTRA_NQ_Eingabe_NT.xlsx")

# Event subfolder pattern: NNN_<duration>h_<return-period>yr
# Duration can be decimal, e.g. 0.25h, 0.5h, 1h, 6h, 72h
re_folder = re.compile(
    r"^(?P<num>\d{3})_(?P<dauer>\d+(?:[.,]\d+)?)h_(?P<yr>\d+)yr$",
    re.IGNORECASE
)

# =============================================================================
# %% Helper functions
# =============================================================================

def ascii_safe(s: str) -> str:
    """Remove umlauts/special chars for safe file names."""
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")


def yr_num(label: str) -> int:
    """Extract integer from label like '100yr' -> 100 (used for sorting)."""
    m = re.search(r"(\d+)\s*yr", str(label), re.IGNORECASE)
    return int(m.group(1)) if m else 10**9


def dauer_num(label: str) -> float:
    """Extract float from label like '0.25 h' or '6 h' (used for sorting)."""
    m = re.search(r"([\d.,]+)", str(label))
    return float(m.group(1).replace(",", ".")) if m else 0.0


def merge_split_underscore_cols(cols):
    """Fix column names split by a space before '_'. e.g. ['EF1', '_1AB'] -> ['EF1_1AB']"""
    fixed = []
    i = 0
    while i < len(cols):
        if (i + 1 < len(cols)
                and cols[i + 1].startswith("_")
                and re.match(r"^[A-Za-z0-9]+$", cols[i])):
            fixed.append(cols[i] + cols[i + 1])
            i += 2
        else:
            fixed.append(cols[i])
            i += 1
    return fixed


def _read_area_km2_from_excel(xlsx_path: Path) -> float:
    """Read catchment area [km2] from 'Eingabe_Parameter' sheet (columns: Parameter / Wert)."""
    if not xlsx_path.exists():
        raise FileNotFoundError(f"Excel file not found: {xlsx_path}")
    dfp = pd.read_excel(xlsx_path, sheet_name="Eingabe_Parameter")
    if not {"Parameter", "Wert"}.issubset(set(map(str, dfp.columns))):
        dfp = dfp.iloc[:, :2]
        dfp.columns = ["Parameter", "Wert"]
    mask = dfp["Parameter"].astype(str).str.lower().str.contains("einzugsgebiet", na=False)
    if not mask.any():
        raise KeyError(f"Catchment area row not found in {xlsx_path}.")
    area_val = pd.to_numeric(dfp.loc[mask, "Wert"], errors="coerce").dropna()
    if area_val.empty:
        raise ValueError(f"Catchment area value in {xlsx_path} is not numeric.")
    return float(area_val.iloc[0])


# =============================================================================
# %% WEL file parser
# =============================================================================

_date_re = re.compile(r"^\d{2}\.\d{2}\.\d{4}$")
_time_re = re.compile(r"^\d{2}:\d{2}$")


def _find_wel_file(folder: Path) -> Path:
    """Search for the WEL file inside an event subfolder (tries lowercase too)."""
    candidates = [
        WEL_NAME,
        WEL_NAME + ".txt",
        WEL_NAME.lower(),
        WEL_NAME.lower() + ".txt",
    ]
    for name in candidates:
        p = folder / name
        if p.exists() and p.is_file():
            return p
    raise FileNotFoundError(
        f"WEL file not found in:\n  {folder}\n"
        f"Searched for: {candidates}"
    )


def parse_wel(path: Path, value_col: str = VALUE_COL, debug: bool = False) -> dict:
    """
    Robustly parse a TALSIM .WEL time-series output file.

    Finds record boundaries by locating the next date+time token pair,
    which prevents column-shifting when values are missing or lines are broken.

    Returns a dict with:
        file    - Path
        q       - numpy array of flow values [m3/s]
        t_h     - numpy array of elapsed time [h from start]
        n       - number of valid records parsed
        columns - list of column names from the header line
    """
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

    # Locate the header line (starts with "Datum_Zeit")
    header_idx = None
    for i, line in enumerate(lines):
        if line.strip().startswith("Datum_Zeit"):
            header_idx = i
            break
    if header_idx is None:
        raise ValueError(f"Header line 'Datum_Zeit' not found in {path}")

    cols = lines[header_idx].split()
    cols = merge_split_underscore_cols(cols)
    if value_col not in cols:
        raise KeyError(
            f"Column '{value_col}' not found in {path.name}.\n"
            f"Available columns: {cols}"
        )

    vpos            = cols.index(value_col)   # column index (including Datum_Zeit)
    expected_tokens = len(cols) + 1           # [Date] [Time] + (len(cols)-1) values

    # Skip unit/comment lines after header until first data line
    start = header_idx + 1
    while start < len(lines):
        s = lines[start].strip()
        if not s or s.startswith("*") or s.lstrip().startswith("-"):
            start += 1
            continue
        tok = s.split()
        if tok and _date_re.match(tok[0]):
            break
        start += 1

    def find_dt_start(buffer, start_at=0):
        """Return index in buffer where the next Date+Time pair begins."""
        for j in range(start_at, len(buffer) - 1):
            if _date_re.match(buffer[j]) and _time_re.match(buffer[j + 1]):
                return j
        return None

    times, values = [], []
    bad_len = 0
    bad_val = 0
    total_rec = 0
    buf = []

    for line in lines[start:]:
        if not line.strip() or line.strip().startswith("*"):
            continue
        buf.extend(line.split())

        while True:
            s0 = find_dt_start(buf)
            if s0 is None:
                if len(buf) > 10 * expected_tokens:
                    buf = buf[-expected_tokens:]
                break

            if s0 > 0:
                buf = buf[s0:]   # discard tokens before the date

            s1 = find_dt_start(buf, start_at=2)
            if s1 is None:
                if len(buf) < expected_tokens:
                    break
                end = expected_tokens
            else:
                end = s1 if s1 < expected_tokens else expected_tokens
                if len(buf) < end:
                    break

            rec = buf[:end]
            buf = buf[end:]
            total_rec += 1

            if len(rec) != expected_tokens:
                bad_len += 1
                continue

            dstr, tstr = rec[0], rec[1]
            try:
                ts = dt.datetime.strptime(dstr + " " + tstr, "%d.%m.%Y %H:%M")
            except Exception:
                ts = None

            raw = rec[2:][vpos - 1].replace(",", ".")
            try:
                val = float(raw)
            except Exception:
                bad_val += 1
                continue

            times.append(ts)
            values.append(val)

    q = np.asarray(values, dtype=float)

    # Build time axis in hours from the first timestamp
    if times and all(t is not None for t in times):
        t0  = times[0]
        t_h = np.array([(t - t0).total_seconds() / 3600.0 for t in times], dtype=float)
    else:
        t_h = np.arange(len(q), dtype=float)

    if debug:
        print(
            f"  [DEBUG] {path.name}: "
            f"total={total_rec}, valid={len(q)}, "
            f"bad_len={bad_len}, bad_val={bad_val}, "
            f"cols={len(cols)}"
        )

    return {"file": path, "q": q, "t_h": t_h, "n": len(q), "columns": cols}


# =============================================================================
# %% Plotting
# =============================================================================

def plot_waves_for_year(year_label: str, dct: dict, plots_dir: Path, folder_tag: str) -> Path:
    """
    Plot inflow hydrographs for all storm durations of one return period.
    Thesis-quality scientific style. Saves one PNG per return period.
    """
    plt.rcParams.update({
        'font.family':      'monospace',
        'font.size':         9,
        'axes.titlesize':   10,
        'axes.labelsize':    9,
        'xtick.labelsize':   8,
        'ytick.labelsize':   8,
        'legend.fontsize':   8,
        'axes.linewidth':    0.8,
        'grid.linewidth':    0.5,
        'grid.alpha':        0.4,
        'grid.linestyle':    '--',
        'axes.grid':         True,
        'axes.axisbelow':    True,
    })

    dauer_list = sorted(dct.keys(), key=float)
    n          = len(dauer_list)

    # -------------------------------------------------------------------------
    # Peak-ranked coloring:
    #   - The duration with the HIGHEST peak always gets red (#e00000)
    #   - All other durations are colored by their position in a
    #     blue → green → yellow → orange ramp (red excluded)
    #   - This makes the critical duration instantly visible
    # -------------------------------------------------------------------------
    import matplotlib.colors as mcolors

    # Step 1: find which duration has the highest peak
    peak_per_dauer = {
        d: float(np.nanmax(dct[d]["q"])) if len(dct[d]["q"]) > 0 else 0.0
        for d in dauer_list
    }
    critical_dauer = max(peak_per_dauer, key=peak_per_dauer.get)

    # Step 2: ramp for non-critical lines (blue → green → yellow → orange)
    NON_CRIT_RAMP = mcolors.LinearSegmentedColormap.from_list(
        "non_crit",
        [
            "#08306b",   # deep blue
            "#2196c8",   # cyan
            "#1a6b2f",   # dark green
            "#78c44a",   # yellow-green
            "#f5e400",   # yellow
            "#f57c00",   # orange
            "#7b3a10",   # brown
        ]
    )
    non_crit_list = [d for d in dauer_list if d != critical_dauer]
    nc = len(non_crit_list)

    # Step 3: assign colors — evenly spaced across ramp for non-critical
    colors = {}
    for i, d in enumerate(non_crit_list):
        p = i / max(nc - 1, 1)
        colors[d] = NON_CRIT_RAMP(p)
    colors[critical_dauer] = "#e00000"   # always red

    fig, ax = plt.subplots(figsize=(6.5, 4.5))

    qmax_global = 0.0
    for k, dauer in enumerate(dauer_list):
        q = dct[dauer]["q"]
        t = dct[dauer]["t_h"]
        qmax_global = max(qmax_global, float(np.nanmax(q)) if len(q) > 0 else 0)
        is_crit = (dauer == critical_dauer)
        ax.plot(
            t, q,
            color     = colors[dauer],
            linestyle = "-",
            linewidth = 1.2 if is_crit else 1.0,   # critical line slightly thicker
            alpha     = 0.85 if is_crit else 0.82,
            label     = f"{dauer:g} h  ★" if is_crit else f"{dauer:g} h",
            zorder    = 50 if is_crit else k + 2,   # critical always on top
        )

    # Mark peak of each hydrograph with a small triangle
    for k, dauer in enumerate(dauer_list):
        q = dct[dauer]["q"]
        t = dct[dauer]["t_h"]
        if len(q) == 0:
            continue
        is_crit = (dauer == critical_dauer)
        idx = int(np.nanargmax(q))
        ax.plot(t[idx], q[idx],
                marker="^", markersize=6 if is_crit else 4,
                color=colors[dauer], zorder=60 if is_crit else k + 10,
                markeredgewidth=0.5, markeredgecolor="white")

    # Axes limits and ticks
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0, top=qmax_global * 1.12)
    ax.xaxis.set_major_locator(MaxNLocator(10, integer=False))
    ax.tick_params(axis="x", which="major", length=5)
    ax.tick_params(axis="y", which="major", length=5)

    ax.set_xlabel("Time [h] (from event start)")
    ax.set_ylabel(f"Q [m\u00b3/s]")
    ax.set_title(
        f"Design Storm Hydrographs  \u2014  Return Period (T)\u2009=\u2009{year_label.replace('yr', ' yr')}",
        pad=8
    )

    # Legend: two columns, clean box
    leg = ax.legend(
        title="Storm Duration",
        ncol=2,
        loc="upper right",
        framealpha=0.92,
        edgecolor="0.7",
        borderpad=0.6,
        labelspacing=0.3,
        handlelength=1.8,
    )
    leg.get_title().set_fontsize(7)
    leg.get_title().set_fontstyle("italic")

    # Subtle footer
    # Clean spines: only left + bottom
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)

    fig.tight_layout()
    out_png = plots_dir / f"{ascii_safe(folder_tag)}_Hydrographs_{ascii_safe(year_label)}.png"
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return out_png

def build_excel(by_year: dict, found_years: list, out_dir: Path, base_name: str) -> Path:
    """
    Write one Excel file with one sheet per return period.
    Columns: Time [h] + one column per storm duration.
    """
    excel_path = out_dir / f"{ascii_safe(base_name)}_Results.xlsx"
    with pd.ExcelWriter(excel_path, engine="openpyxl") as xw:
        for ylab in sorted(found_years, key=yr_num):
            dct = by_year[ylab]
            dfs = []
            for d in sorted(dct.keys(), key=float):
                t = np.round(dct[d]["t_h"].astype(float), 6)
                q = dct[d]["q"].astype(float)
                dfs.append(pd.DataFrame({"Time [h]": t, f"{float(d):g} h": q}))

            dfm = reduce(
                lambda left, right: pd.merge(left, right, on="Time [h]", how="outer"),
                dfs
            )
            dfm = dfm.sort_values("Time [h]").reset_index(drop=True)
            dfm.to_excel(xw, index=False, sheet_name=ylab[:31])
    return excel_path


# =============================================================================
# %% Summary table
# =============================================================================

def build_summary(by_year: dict, found_years: list, out_dir: Path,
                  base_name: str, excel_param_path=EXCEL_PARAM_PATH):
    """
    Tab-separated summary:
        Return Period | Qmax [m3/s] | Specific Runoff [l/(km2 s)] | Critical Duration [h]
    """
    txt_path = out_dir / f"{ascii_safe(base_name)}_Summary.txt"

    area_km2 = None
    if excel_param_path is not None:
        try:
            area_km2 = _read_area_km2_from_excel(excel_param_path)
        except Exception as e:
            print(f"  [WARNING] Could not read catchment area: {e}")

    rows = []
    for ylab in found_years:
        dct = by_year[ylab]
        max_per_d = {d: float(np.nanmax(dct[d]["q"])) for d in dct if len(dct[d]["q"]) > 0}
        if not max_per_d:
            continue
        d_star = max(max_per_d, key=max_per_d.get)
        qmax   = max_per_d[d_star]
        spende = (1000.0 * qmax / area_km2) if (area_km2 and area_km2 > 0) else float("nan")

        rows.append({
            "Return Period":               ylab,
            "Qmax [m3/s]":                 qmax,
            "Specific Runoff [l/(km2 s)]": spende,
            "Critical Duration [h]":       float(d_star),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(
            "Summary table is empty - no valid Q series were read.\n"
            "Set debug=True in parse_wel() and check bad_len / bad_val counters."
        )

    df["_yr"] = df["Return Period"].map(yr_num)
    df = df.sort_values("_yr").drop(columns=["_yr"]).reset_index(drop=True)
    df.to_csv(txt_path, sep="\t", index=False, encoding="utf-8", float_format="%.3f")
    return txt_path, df


# =============================================================================
# %% Peak flow matrix
# =============================================================================

def build_peak_matrix(by_year: dict, found_years: list, out_dir: Path, base_name: str):
    """
    Tab-separated peak flow matrix:
        Rows    = storm durations [h]
        Columns = return periods [yr]
    """
    txt_path      = out_dir / f"{ascii_safe(base_name)}_PeakMatrix.txt"
    all_durations = sorted({d for y in by_year for d in by_year[y]}, key=float)
    peaks         = pd.DataFrame(index=all_durations, columns=found_years, dtype=float)

    for ylab in found_years:
        for d, data in by_year[ylab].items():
            if len(data["q"]) > 0:
                peaks.at[d, ylab] = float(np.nanmax(data["q"]))

    peaks.index      = [f"{float(d):g}" for d in peaks.index]
    peaks.index.name = "Duration [h]"
    peaks = peaks.reindex(columns=sorted(found_years, key=yr_num))
    peaks.to_csv(txt_path, sep="\t", float_format="%.2f", na_rep="", encoding="utf-8")
    return txt_path, peaks


# =============================================================================
# %% Main processing function
# =============================================================================

def run_for_root(root_dir: Path):
    """
    Scan all event subfolders in root_dir, parse their WEL files,
    and produce plots, Excel, summary table, and peak matrix.

    Subfolder pattern : NNN_<duration>h_<return-period>yr
    Duration          : any decimal value, e.g. 0.25h, 0.5h, 1h, 6h, 72h
    Return period     : any integer in JAEHRLICHKEITEN_YR
    WEL file          : must match WEL_NAME (set at top of script)

    Outputs are saved to:  <root_dir>/results/
    """
    if not root_dir.exists():
        raise FileNotFoundError(f"Folder not found: {root_dir}")

    out_dir   = root_dir / "results"
    plots_dir = out_dir  / "plots"
    out_dir.mkdir(exist_ok=True)
    plots_dir.mkdir(parents=True, exist_ok=True)

    by_year   = {}
    processed = 0
    skipped   = 0

    subfolders = sorted([p for p in root_dir.iterdir() if p.is_dir()])
    if not subfolders:
        raise FileNotFoundError(f"No subfolders found in:\n  {root_dir}")

    for sf in subfolders:
        m = re_folder.match(sf.name)
        if not m:
            skipped += 1
            continue

        # Parse duration (supports decimals like 0.25) and return period
        dauer_h    = float(m.group("dauer").replace(",", "."))
        yr         = int(m.group("yr"))
        year_label = f"{yr}yr"

        # Skip return periods outside the expected list
        if yr not in JAEHRLICHKEITEN_YR:
            skipped += 1
            continue

        try:
            wel_path = _find_wel_file(sf)
        except FileNotFoundError as e:
            print(f"  [WARNING] {e}")
            skipped += 1
            continue

        parsed = parse_wel(wel_path, value_col=VALUE_COL, debug=False)
        by_year.setdefault(year_label, {})[dauer_h] = parsed
        processed += 1

    print(f"\n[{root_dir.name}]  processed: {processed}  |  skipped: {skipped}")

    found_years = sorted(by_year.keys(), key=yr_num)
    if not found_years:
        raise RuntimeError(
            f"No valid event subfolders were processed in:\n  {root_dir}\n\n"
            f"Expected subfolder pattern : NNN_<duration>h_<return-period>yr\n"
            f"Examples                   : 001_0.25h_100yr  |  006_6h_100yr  |  143_72h_5yr\n"
            f"Return periods accepted    : {JAEHRLICHKEITEN_YR}"
        )

    # Show all durations found (no fixed list needed)
    all_durations = sorted({d for y in by_year for d in by_year[y]}, key=float)
    print(f"[{root_dir.name}]  Return periods : {found_years}")
    print(f"[{root_dir.name}]  Durations [h]  : {[f'{d:g}' for d in all_durations]}")

    # Plots
    for ylab in found_years:
        png = plot_waves_for_year(ylab, by_year[ylab], plots_dir, folder_tag=root_dir.name)
        print(f"  Plot : {png.name}")

    # Excel
    excel_path = build_excel(by_year, found_years, out_dir, base_name=root_dir.name)
    print(f"  Excel: {excel_path.name}")

    # Summary
    summary_path, df_summary = build_summary(
        by_year, found_years, out_dir,
        base_name=root_dir.name,
        excel_param_path=EXCEL_PARAM_PATH
    )
    print(f"  Summary: {summary_path.name}")

    # Peak matrix
    peaks_path, df_peaks = build_peak_matrix(by_year, found_years, out_dir, base_name=root_dir.name)
    print(f"  Peak matrix: {peaks_path.name}")

    return df_summary, df_peaks, excel_path, summary_path, peaks_path


# =============================================================================
# %% Run
# =============================================================================

print(f"\n{'='*60}")
print(f"Processing: {MAIN_FOLDER}")
print(f"WEL file  : {WEL_NAME}")
print("="*60)

df_sum, df_pk, xlsx, txt_sum, txt_pk = run_for_root(MAIN_FOLDER)

print("\n--- Summary ---")
print(df_sum.to_string(index=False))
print("\n--- Peak Matrix ---")
print(df_pk.to_string())